# WorldSplat-Prune — experiment runner

Runs the experiment manifest (upstream **3DGS baseline** + this checkout's base
trainer) and extracts metrics, using the `experiments/` harness. The harness
clones external codebases, installs the right CUDA rasterizer, and **swaps
rasterizers** between methods (this fork's `dr_aa` rasterizer vs upstream's).
See `experiments/README.md`.

The **post-reconstruction pruning method** and its ablation manifest are added
separately (see `docs/worldsplat_prune_spec.md`); this notebook currently drives
the training baselines that produce the models to prune.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Get this repo + dependencies

Clones/refreshes this WorldSplat-Prune checkout (which contains `train.py`, the `experiments/`
harness, and the `eval_worldsplat.py` evaluator) and initializes the two training
submodules. Python deps the repo imports but the Colab runtime does not ship
(`plyfile`, ...) are installed here; the CUDA rasterizers themselves are built
by the harness per method (step 6). The evaluator uses the bundled
`lpipsPyTorch`, so no extra pip deps for metrics.

In [ ]:
import os

REPO_URL = "https://github.com/satyaba/SGDC-3DGS.git"
REPO_DIR = "/content/SGDC-3DGS"
BRANCH = "feature/worldsplat-prune"

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch --all -q
    !git -C {REPO_DIR} checkout {BRANCH} -q
    !git -C {REPO_DIR} pull --ff-only -q

os.chdir(REPO_DIR)

# Training submodules: rasterizer (dr_aa) + simple-knn.
!git submodule update --init --recursive submodules/diff-gaussian-rasterization submodules/simple-knn
!git -C submodules/diff-gaussian-rasterization log --oneline -1

# Python deps the repo imports but the Colab runtime does not ship.
# (plyfile is imported by scene/gaussian_model.py; without it training exits 1.)
!pip install -q plyfile opencv-python joblib

print("cwd:", os.getcwd())

## 3. GPU / CUDA check

The rasterizer is compiled against a CUDA toolkit that must match `torch.version.cuda`.

In [ ]:
!nvidia-smi

import re, shutil, subprocess, warnings
import torch

print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | "
          f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

def _nvcc_cuda():
    exe = shutil.which("nvcc")
    if exe is None:
        return None
    out = subprocess.run([exe, "--version"], capture_output=True, text=True).stdout
    m = re.search(r"release (\d+)\.(\d+)", out)
    return f"{m.group(1)}.{m.group(2)}" if m else None

torch_cuda, nvcc_cuda = torch.version.cuda, _nvcc_cuda()
print(f"  torch built for CUDA {torch_cuda} | nvcc {nvcc_cuda or 'NOT FOUND'}")
if nvcc_cuda is None:
    warnings.warn("nvcc not found — CUDA extensions cannot be built.")
elif torch_cuda is None:
    warnings.warn("This PyTorch build has no CUDA; extensions cannot be built.")
elif torch_cuda.split(".")[0] != nvcc_cuda.split(".")[0]:
    raise RuntimeError(f"CUDA major mismatch: torch {torch_cuda} vs nvcc {nvcc_cuda}.")
else:
    print("  OK: toolchain consistent (nvcc matches torch).")

## 4. Configure paths

`experiments/runs.py` assumes `<HOME_DIR>/Dataset/<dataset>` with `images/`,
`sparse/0/`; `<dataset>/semantic_class/` (GT class labels) enables the
band/interior metrics.

In [ ]:
import os
from pathlib import Path

HOME_DIR = "/content/drive/MyDrive/Magister/Thesis/Script"
DATASET_NAME = "office_0/Sequence_1"

os.environ["HOME_DIR"] = HOME_DIR
os.environ["DATASET_ROOT"] = str(Path(HOME_DIR) / "Dataset")

DATASET_DIR = Path(os.environ["DATASET_ROOT"]) / DATASET_NAME
os.environ["DATASET_DIR"] = str(DATASET_DIR)

for k in ("HOME_DIR", "DATASET_ROOT", "DATASET_DIR"):
    print(f"{k} = {os.environ[k]}")
# semantic_class/ is optional (needed only for band/interior metrics)

## 5. Inspect the run manifest

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())          # make `experiments` importable from repo root

from experiments.runs import RUNS, run_by_id, run_all, run_sanity

for r in RUNS.values():
    kind = "sanity" if r.sanity else "full"
    print(f"[{kind}] {r.run_id:26s} method={r.method:13s} iters={r.iterations:<6d} {r.notes}")

## 6. Sanity check (run this first)

Runs the short 3 000-iteration versions and extracts their metrics. 3 000 iters
crosses the first densification (500), hits the opacity reset (3000) and saves —
so it exercises the whole pipeline (train → rasterizer swap → eval) before the
full 30 000-iteration runs. Preview with `run_sanity(dry=True)`.


In [ ]:
# Short end-to-end smoke test (also builds/swaps rasterizers under the hood):
run_sanity()

In [ ]:
from experiments.metrics import extract_sanity

extract_sanity()

## 7. Full training (30k)

`run_all()` trains every non-sanity run. If you already ran the sanity check,
the rasterizer for the current method is likely cached (no rebuild). Run one at
a time with `run_by_id(...)`.

In [ ]:
run_all()

# --- or one at a time ---
# run_by_id("vanilla_30k")   # upstream graphdeco baseline
# run_by_id("base_30k")      # this checkout's base trainer

# --- per-run hyperparameter overrides (appended to the spec's args) ---
# run_by_id("base_30k", extra="--lambda_dssim 0.3")
# run_by_id("base_30k", extra=["--densify_until_iter", "10000"])

# --- a sweep: tag gives each call its own output dir ---
# for fi in (2.0, 3.0, 5.0):
#     run_by_id("base_30k", tag=f"fi{fi}", extra=f"--lambda_dssim {fi}")


## 8. Extract metrics

Computes train/test PSNR / SSIM / LPIPS plus boundary / interior test PSNR, Gaussian count and timings; appends a row per run to `$HOME_DIR/output/metrics.csv` (same code path for pruned and vanilla models).

In [ ]:
from experiments.metrics import extract_all, extract_run

extract_all()
# extract_run("base_30k")
# extract_run("base_30k", tag="fi3.0")   # a tagged sweep point

## 9. Results

In [ ]:
import os
import pandas as pd

csv_path = os.path.join(os.environ["HOME_DIR"], "output", "metrics.csv")
print(csv_path)
pd.read_csv(csv_path)

## 10. (Optional) Ad-hoc evaluation

Score an arbitrary model dir with the boundary-restricted evaluator directly.

In [ ]:
from experiments.metrics import extract_one
extract_one('$HOME_DIR/output/base_30k', iteration=30000, name='base@30k',
            run_id='base_30k', csv_path=None)   # csv_path=None -> print only

# richer: band vs interior detail, no CSV
# !python eval_worldsplat.py -m "$HOME_DIR/output/base_30k" --iteration 30000 --boundary_band 4

## Notes / next steps

- Full method/math and the post-reconstruction pruning pipeline:
  `docs/worldsplat_prune_spec.md`. Harness details: `experiments/README.md`.
- `worldsplat_prune.eval_boundary` remains available for quick interactive checks.
- The pruning-method ablation manifest (arms over the operator's `s`, `delta`,
  protect set, ...) is added to `experiments/runs.py` separately.
